<a href="https://www.kaggle.com/code/mufidpanhalkar/titanic-survival-prediction-xgboost?scriptVersionId=316048184" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## 🧾 1. Introduction

The Titanic dataset is a classic binary classification problem where the objective is to predict whether a passenger survived or not based on features like age, class, gender, and more.

This notebook focuses on:

Strong feature engineering
Efficient preprocessing
High-performance model using XGBoost

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# 📦 2. Import Libraries

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

# 📂 3. Load Dataset

In [3]:
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

train.head()

Train Shape: (891, 12)
Test Shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 🔗 4. Combine Data for Preprocessing

In [4]:
test_ids = test['PassengerId']

full = pd.concat([train, test], sort=False).reset_index(drop=True)

# 🧠 5. Feature Engineering

Feature engineering is the most critical step for improving model performance.

# 🔹 5.1 Extract Title from Name

In [5]:
full['Title'] = full['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
full['Title'] = full['Title'].replace(rare_titles, 'Rare')
full['Title'] = full['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

# 🔹 5.2 Family-Based Features

In [6]:
full['FamilySize'] = full['SibSp'] + full['Parch'] + 1
full['IsAlone'] = (full['FamilySize'] == 1).astype(int)

# 🔹 5.3 Age & Fare Processing

In [7]:
full['Age'] = full['Age'].fillna(full['Age'].median())
full['Fare'] = full['Fare'].fillna(full['Fare'].median())

full['AgeBin'] = pd.cut(full['Age'], 5, labels=False)
full['FareBin'] = pd.qcut(full['Fare'], 4, labels=False)

# 🔹 5.4 Handle Missing Embarked

In [8]:
full['Embarked'] = full['Embarked'].fillna(full['Embarked'].mode()[0])

# 🔹 5.5 Drop Unnecessary Columns

In [9]:
full = full.drop(['Name', 'Ticket', 'Cabin'], axis=1, errors='ignore')

# 🔹 5.6 One-Hot Encoding

In [10]:
full = pd.get_dummies(full, columns=['Sex','Embarked','Title'], drop_first=True)

# ✂️ 6. Split Back into Train & Test

In [11]:
train_df = full[~full['Survived'].isnull()]
test_df = full[full['Survived'].isnull()]

X = train_df.drop(['Survived','PassengerId'], axis=1)
y = train_df['Survived']

X_test = test_df.drop(['Survived','PassengerId'], axis=1)

# 🧪 7. Train-Validation Split

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 🚀 8. Model Training (XGBoost)

In [13]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

# 🔁 9. Retrain on Full Dataset

In [16]:
model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

# 📊 10. Model Evaluation

In [17]:
val_preds = model.predict(X_val)
accuracy = accuracy_score(y_val, val_preds)

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.9217877094972067


# 📤 11. Generate Predictions

In [18]:
predictions = model.predict(X_test)

# 📁 12. Create Submission File

In [19]:
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": predictions.astype(int)
})

submission.to_csv("submission.csv", index=False)

print("✅ Submission file created successfully!")

✅ Submission file created successfully!


## 🏁 Conclusion

In this notebook, we:

* Applied feature engineering techniques like Title extraction and FamilySize
* Handled missing values effectively
* Used XGBoost, a powerful ensemble algorithm
* Built a clean pipeline ready for Kaggle submission

## 🔥 Future Improvements
* Hyperparameter tuning (Optuna)
* Cross-validation (Stratified K-Fold)
* Model ensembling (XGBoost + LightGBM)
* Advanced features (Cabin deck, ticket groups)